# DynaLoRA: extended experiments (Colab, GPU)

Runs the full level-2 pipeline: trains LoRA adapters for each corpus configuration, generates under DynaLoRA, weight-space scaling, classifier-free guidance and the temperature baselines, and evaluates with external perplexity, a domain classifier trained on held-out human text, a sentiment classifier and lexical metrics.

Runtime: select a GPU (T4 is enough). Expected total time: about 2 to 3 hours for GPT-2 small on five configurations plus Qwen2.5-0.5B on one.

In [ ]:
!nvidia-smi -L
!git clone https://github.com/alexroehrs/dynalora.git
%cd /content/dynalora
!pip install -q -r requirements.txt
!pip uninstall -y -q torchao  # Colab ships torchao 0.10, which the current peft rejects

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT='/content/drive/MyDrive/dynalora_results'
os.makedirs(OUT, exist_ok=True)
print(OUT)

## 1. GPT-2 small, five corpus configurations
`imdb_neg` reproduces the original recipe (negative reviews only, lr 5e-6, unmasked padding). The others use balanced IMDB, balanced IMDB without HTML tags, Yelp and PubMed abstracts, with the standard LoRA learning rate and masked padding. Results are written incrementally, so the cell can be re-run after a disconnection and skips what is done.

In [ ]:
!python scripts/run_all.py --out $OUT/gpt2 --models gpt2 --configs imdb_neg imdb_bal imdb_bal_nohtml yelp pubmed --n_samples 20

## 2. A current small model: Qwen2.5-0.5B on balanced IMDB

In [ ]:
!python scripts/run_all.py --out $OUT/qwen --models Qwen/Qwen2.5-0.5B --configs imdb_bal --n_samples 20

## 3. Level-1 metrics over the original released generations

In [ ]:
!python scripts/level1_original_generations.py --out $OUT/level1

## 4. Package the results

In [ ]:
!cd $OUT && zip -qr /content/dynalora_results.zip . -x '*/adapters/*'
from google.colab import files
files.download('/content/dynalora_results.zip')